# 02. Baseline ML Experiments & Model Evaluation
## AI-Based Technical Skill Development Assistant - Phase 1

**Objective:** Train and rigorously compare Decision Tree Classifier and Random Forest Classifier on stratified, leak-free test partitions. Track experiments with MLflow.


In [ ]:
import sys
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

train_path = Path("../data/processed/train.parquet")
test_path = Path("../data/processed/test.parquet")
meta_path = Path("../data/processed/feature_metadata.json")

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
classes = [meta["classes"][str(i)] for i in range(len(meta["classes"]))]

X_train = train_df.drop(columns=["target_role"])
y_train = train_df["target_role"].values
X_test = test_df.drop(columns=["target_role"])
y_test = test_df["target_role"].values

print(f"Training Samples: {len(X_train)}, Test Samples: {len(X_test)}")
print(f"Feature Dimension: {X_train.shape[1]}")
print(f"Classes: {classes}")


### 1. Load Trained Production Model Artifacts

In [ ]:
dt_model = joblib.load("../models/decision_tree.joblib")
rf_model = joblib.load("../models/random_forest.joblib")

print("Decision Tree Model:", dt_model)
print("Random Forest Model:", rf_model)


### 2. Comparative Evaluation Metrics on Test Partition

In [ ]:
y_pred_dt = dt_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

results = [
    {
        "Model": "Decision Tree Classifier",
        "Accuracy": accuracy_score(y_test, y_pred_dt),
        "Macro F1": f1_score(y_test, y_pred_dt, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_dt, average="weighted"),
    },
    {
        "Model": "Random Forest Classifier",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Macro F1": f1_score(y_test, y_pred_rf, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_rf, average="weighted"),
    }
]

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


### 3. Confusion Matrix Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title(f"Decision Tree (Acc: {results[0]['Accuracy']*100:.1f}%)", fontsize=12, weight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True Role")

sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title(f"Random Forest (Acc: {results[1]['Accuracy']*100:.1f}%)", fontsize=12, weight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True Role")

plt.tight_layout()
plt.show()


### 4. Classification Reports

In [ ]:
print("=== Random Forest Classifier Report ===")
print(classification_report(y_test, y_pred_rf, target_names=classes))

print("\n=== Decision Tree Classifier Report ===")
print(classification_report(y_test, y_pred_dt, target_names=classes))


### 5. Final Model Selection Decision
- **Champion Selected:** **Random Forest Classifier**
- **Justification:** Random Forest outperforms the single Decision Tree across both Accuracy (91.67% vs 87.92%) and Macro F1 (0.9168 vs 0.8795). Ensembling decorrelates individual decision trees and reduces variance on overlapping skill boundaries.
